# Bygning med Meta-familie modellerne 

## Introduktion 

Denne lektion vil dække: 

- Udforskning af de to hoved Meta-familiemodeller - Llama 3.1 og Llama 3.2 
- Forstå brugssager og scenarier for hver model 
- Kodeeksempel for at vise de unikke funktioner ved hver model 


## Meta-familien af modeller 

I denne lektion vil vi udforske 2 modeller fra Meta-familien eller "Llama-flokken" - Llama 3.1 og Llama 3.2 

Disse modeller findes i forskellige varianter og er tilgængelige i [Microsoft Foundry Models kataloget](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst).

> **Bemærk:** GitHub Models udfases ved udgangen af juli 2026. Her er flere detaljer om brug af [Microsoft Foundry Models](https://learn.microsoft.com/en-us/azure/ai-foundry/model-inference/overview?WT.mc_id=academic-105485-koreyst) til at prototype med AI-modeller.

Modelvarianter: 
- Llama 3.1 - 70B Instruct 
- Llama 3.1 - 405B Instruct 
- Llama 3.2 - 11B Vision Instruct 
- Llama 3.2 - 90B Vision Instruct 

*Bemærk: Llama 3 er også tilgængelig i Microsoft Foundry Models, men vil ikke blive dækket i denne lektion*



## Llama 3.1 

Med 405 milliarder parametre passer Llama 3.1 ind under kategorien open source LLM. 

Modellen er en opgradering af den tidligere udgivelse Llama 3 ved at tilbyde: 

- Større kontekstvindue - 128k tokens vs 8k tokens 
- Større maks output tokens - 4096 vs 2048 
- Bedre flersproget support - på grund af stigning i træningstokens 

Disse gør det muligt for Llama 3.1 at håndtere mere komplekse brugssager, når man bygger GenAI-applikationer, herunder: 
- Native Function Calling - muligheden for at kalde eksterne værktøjer og funktioner uden for LLM-arbejdsgangen
- Bedre RAG-ydeevne - på grund af det større kontekstvindue 
- Syntetisk datagenerering - muligheden for at skabe effektiv data til opgaver såsom finjustering 



### Indbygget Funktionskald 

Llama 3.1 er finjusteret til at være mere effektiv til at lave funktions- eller værktøjskald. Den har også to indbyggede værktøjer, som modellen kan identificere som nødvendige at bruge baseret på brugerens prompt. Disse værktøjer er: 

- **Brave Search** - Kan bruges til at få opdaterede oplysninger som vejret ved at udføre en web-søgning 
- **Wolfram Alpha** - Kan bruges til mere komplekse matematiske beregninger, så du ikke behøver at skrive dine egne funktioner. 

Du kan også lave dine egne brugerdefinerede værktøjer, som LLM kan kalde. 

I kodeeksemplet nedenfor: 

- Definerer vi de tilgængelige værktøjer (brave_search, wolfram_alpha) i systemprompten. 
- Sender en brugerprompt, der spørger om vejret i en bestemt by. 
- LLM vil svare med et værktøjskald til Brave Search-værktøjet, som vil se sådan ud `<|python_tag|>brave_search.call(query="Stockholm weather")` 

*Bemærk: Dette eksempel laver kun værktøjskaldet, hvis du vil have resultaterne, skal du oprette en gratis konto på Brave API-siden og definere funktionen selv` 


In [ ]:
%pip install azure-core
%pip install azure-ai-inference

In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import AssistantMessage, SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]
model_name = "Meta-Llama-3.1-405B-Instruct"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


tool_prompt=f"""
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Environment: ipython
Tools: brave_search, wolfram_alpha
Cutting Knowledge Date: December 2023
Today Date: 23 July 2024

You are a helpful assistant<|eot_id|>
"""

messages = [
    SystemMessage(content=tool_prompt),
    UserMessage(content="What is the weather in Stockholm?"),

]

response = client.complete(messages=messages, model=model_name)

print(response.choices[0].message.content)






### Llama 3.2 

På trods af at være en LLM, har Llama 3.1 en begrænsning i forhold til multimodalitet. Det vil sige, at kunne bruge forskellige typer input såsom billeder som prompts og give svar. Denne evne er en af hovedfunktionerne i Llama 3.2. Disse funktioner inkluderer også: 

- Multimodalitet - har evnen til at evaluere både tekst- og billedprompter 
- Små til mellemstore størrelsesvarianter (11B og 90B) - dette giver fleksible implementeringsmuligheder, 
- Tekst-only varianter (1B og 3B) - dette gør det muligt at implementere modellen på edge/mobilenheder og giver lav latenstid 

Den multimodale understøttelse repræsenterer et stort skridt i verden af open source-modeller. Kodeeksemplet nedenfor tager både et billede og tekstprompt for at få en analyse af billedet fra Llama 3.2 90B. 

### Multimodal support med Llama 3.2


In [ ]:
%pip install azure-core
%pip install azure-ai-inference

In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import (
    SystemMessage,
    UserMessage,
    TextContentItem,
    ImageContentItem,
    ImageUrl,
    ImageDetailLevel,
)
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]
model_name = "Llama-3.2-90B-Vision-Instruct"


client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)

response = client.complete(
    messages=[
        SystemMessage(
            content="You are a helpful assistant that describes images in details."
        ),
        UserMessage(
            content=[
                TextContentItem(text="What's in this image?"),
                ImageContentItem(
                    image_url=ImageUrl.load(
                        image_file="sample.jpg",
                        image_format="jpg",
                        detail=ImageDetailLevel.LOW)
                ),
            ],
        ),
    ],
    model=model_name,
)

print(response.choices[0].message.content)


## Læringen stopper ikke her, fortsæt rejsen

Efter at have gennemført denne lektion kan du tjekke vores [Generative AI Learning collection](https://aka.ms/genai-collection?WT.mc_id=academic-105485-koreyst) for at fortsætte med at forbedre din viden om Generativ AI!


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfraskrivelse**:
Dette dokument er blevet oversat ved hjælp af AI-oversættelsestjenesten [Co-op Translator](https://github.com/Azure/co-op-translator). Selvom vi bestræber os på nøjagtighed, skal du være opmærksom på, at automatiserede oversættelser kan indeholde fejl eller unøjagtigheder. Det originale dokument på dets oprindelige sprog bør betragtes som den autoritative kilde. For kritisk information anbefales professionel menneskelig oversættelse. Vi påtager os intet ansvar for misforståelser eller fejltolkninger, der opstår som følge af brugen af denne oversættelse.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
